# SLM + QLoRA untuk Indonesian NLP — Proof of Concept
**Model:** Qwen2 1.5B  
**Task evaluasi:** COPAL-ID & IndoCulture(cultural reasoning) + SmSA (sentiment) + IndoNLI
**Metode:** QLoRA fine-tuning dengan subset Bactrian-X ID  
**GPU:** Kaggle T4x2  

Tujuan notebook ini: membuktikan pipeline bekerja dan ada perubahan terukur sebelum vs sesudah fine-tuning.

## Cell 1 — Install Dependencies

In [1]:
print("""
# ============================================================
# Cell 1 — Install Dependencies
# ============================================================
""")


# ============================================================
# Cell 1 — Install Dependencies
# ============================================================



In [2]:
# !pip install -q "numpy<2.0.0"
!pip install torch torchvision torchaudio torchao --index-url https://download.pytorch.org/whl/cu121

!pip install -q transformers==5.6.2
!pip install -q accelerate==1.13.0
!pip install -q peft>=0.19.1
!pip install -q bitsandbytes>=0.49.2
!pip install -q trl==0.19.0
!pip install -q "datasets<4.0.0"
!pip install -q evaluate>=0.4.6
!pip install -q scikit-learn stanza



Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 103.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 2 — Cek GPU

In [3]:
print("""
# ============================================================
# Cell 2 — Cek GPU
# ============================================================
""")


# ============================================================
# Cell 2 — Cek GPU
# ============================================================



In [4]:
import os
import torch
import time
import json
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm

In [5]:
os.makedirs("/kaggle/working/Riset_QLoRA", exist_ok=True)
os.makedirs("/kaggle/working/Riset_QLoRA/qwen-lora-adapter", exist_ok=True)

In [6]:
# Memaksa PyTorch hanya melihat dan menggunakan GPU pertama
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [7]:
import torch

print("=" * 50)
print("GPU CHECK")
print("=" * 50)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"GPU {i}: {props.name} — {vram_gb:.1f} GB VRAM")
    print(f"\nTotal GPU: {torch.cuda.device_count()}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("❌ GPU tidak ditemukan! Pastikan Kaggle Accelerator aktif.")
    print("Pergi ke: Settings → Accelerator → GPU T4 x2")


GPU CHECK
GPU 0: Tesla T4 — 14.6 GB VRAM

Total GPU: 1
CUDA version: 12.8


## Cell 3 — Konfigurasi Eksperimen

In [8]:
print("""
# ============================================================
# Cell 3 — Konfigurasi Eksperimen
# ============================================================
""")


# ============================================================
# Cell 3 — Konfigurasi Eksperimen
# ============================================================



In [9]:
# ============================================================
# KONFIGURASI UTAMA — ubah di sini kalau mau eksperimen lain
# ============================================================
# CONFIG = {
#     # Model
#     "model_name": "Qwen/Qwen2-1.5B",          # model base
#     "output_dir": "/kaggle/working/qwen2-id",  # simpan adapter

#     # QLoRA
#     "lora_r": 16,
#     "lora_alpha": 32,
#     "lora_dropout": 0.05,

#     # Training
#     "max_seq_length": 512,
#     "num_train_epochs": 3,
#     "per_device_train_batch_size": 4,
#     "gradient_accumulation_steps": 4,   # effective batch = 16
#     "learning_rate": 2e-4,
#     "warmup_ratio": 0.03,

#     # Dataset
#     "train_samples": 10000,   # subset Bactrian-X ID — bisa dinaikkan
#     "eval_samples": 200,      # untuk validation loss saat training

#     # Evaluasi
#     "copal_samples": None,    # None = semua sampel (559)
#     "smsa_samples": 200,      # subset SmSA untuk kecepatan
# }

CONFIG = {
    "model_name": "Qwen/Qwen2-1.5B",
    "output_dir": "/kaggle/working/Riset_QLoRA/qwen-lora-adapter",
    # "output_dir": "/content/drive/MyDrive/Riset_QLoRA/qwen-lora-adapter",
    "wiki_dataset_dir": "/kaggle/input/datasets/vorstellenze/wiki-budaya-5100/wiki_budaya_5100.csv",
    # "wiki_dataset_dir": "/content/drive/MyDrive/Riset_QLoRA/wiki_budaya_5100.csv",
    # QLoRA Params
    "lora_r": 32,             # Rank lebih besar untuk kompensasi model kecil (default = 64)
    "lora_alpha": 128,
    "target_modules": "all-linear",

    # =========================================================================
    # 🚨 SAKLAR KENDALI EKSPERIMEN (Ubah di sini untuk menentukan jenis training)
    # =========================================================================
    # 1. Mode Linguistik (C3). Pilihan: "NONE", "NORMALIZATION_ONLY", "REGISTER_ONLY", "MORPH_ONLY","NORM_MORPH",  "FULL_PIPELINE"
    "linguistic_mode": "NORM_MORPH",
    
    # 2. Adaptasi Kosakata (C4). Pilihan: True (Aktif) atau False (Non-aktif)
    "vocab_adaptation": False,
    
    # 3. Nama Folder & Berkas Hasil Ekspor (Sesuaikan dengan fokus eksperimenmu)
    # Contoh nama: "C2_Standard", "C3_Norm_Only", "C4_Vocab_Only", "C5_Joint_Framework"
    "experiment_name": "C3_Norm_Morph",
    # =========================================================================
    
    # Training Params
    "max_seq_length": 512,
    "batch_size": 2,
    "grad_acc_steps": 8,      # Effective batch = 16
    "lr": 2e-5,               # LR rendah untuk cegah forgetting
    "epochs": 3, # default for this xp = 3

    # Dataset Subset
    # "train_samples": 20000,
    # "eval_limit": 300,        # Limit evaluasi per task agar cepat
    "train_samples": 8000,
    "eval_limit": 150,
}
# Panduan Validasi Cepat Kombinasi Saklar Eksperimen:
# - Kondisi C2 (Standard QLoRA)  -> linguistic_mode: "NONE",              vocab_adaptation: False
# - Kondisi C3 (Linguistic Full) -> linguistic_mode: "FULL_PIPELINE",      vocab_adaptation: False
# - Kondisi C4 (Vocab Only)      -> linguistic_mode: "NONE",              vocab_adaptation: True
# - Kondisi C5 (Joint Framework) -> linguistic_mode: "FULL_PIPELINE",      vocab_adaptation: True

print(f"🎯 Nama Eksperimen Aktif   : {CONFIG['experiment_name']}")
print(f"📝 Preprocessing Linguistik: {CONFIG['linguistic_mode']}")
print(f"🚀 Perluasan Kosakata (C4) : {CONFIG['vocab_adaptation']}")

🎯 Nama Eksperimen Aktif   : C3_Norm_Morph
📝 Preprocessing Linguistik: NORM_MORPH
🚀 Perluasan Kosakata (C4) : False


## Cell 4 — Load Model & Tokenizer (4-bit)

In [10]:
print("""
# ============================================================
# Cell 4 — Load Model & Tokenizer (4-bit)
# ============================================================
""")


# ============================================================
# Cell 4 — Load Model & Tokenizer (4-bit)
# ============================================================



In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map={"": "cuda:0"}, # Tetap di GPU 0 untuk efisiensi
    dtype=torch.float16,
    trust_remote_code=True
)

print("✅ Load Tokenizer & Model")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

✅ Load Tokenizer & Model


## Cell 4.5 — SETUP LINGUISTIC PREPROCESSING (STANZA)

In [12]:
import re
import stanza
from collections import Counter

In [13]:


# Download dan Load model Stanza (Hanya dieksekusi sekali)
print("⏳ Mengunduh dan memuat model Stanza Indonesia...")
stanza.download('id', processors='tokenize,pos,lemma')
nlp_id = stanza.Pipeline(lang='id', processors='tokenize,pos,lemma', use_gpu=True, verbose=False)
print("✅ Stanza siap digunakan!")

# --- KAMUS NORMALISASI ---
# Kamus Alay — subset yang paling umum
# Sumber: Salsabila et al. (IALP 2018)
KAMUS_ALAY = {
    # Kata ganti orang
    "gw": "saya", "gue": "saya", "gua": "saya",
    "lo": "kamu", "lu": "kamu", "elo": "kamu",
    "dia": "dia", "dy": "dia", "dya": "dia",

    # Kata umum yang sering disingkat
    "yg": "yang", "yng": "yang",
    "dgn": "dengan", "dg": "dengan",
    "tdk": "tidak", "ga": "tidak", "gak": "tidak",
    "nggak": "tidak", "ngga": "tidak", "enggak": "tidak",
    "udah": "sudah", "udh": "sudah", "sdh": "sudah",
    "blm": "belum", "blum": "belum",
    "bgt": "banget", "bngt": "banget",
    "emg": "memang", "emang": "memang",
    "bkn": "bukan", "bkn": "bukan",
    "krn": "karena", "karna": "karena",
    "klo": "kalau", "klu": "kalau", "kl": "kalau",
    "tp": "tapi", "tpi": "tapi",
    "jd": "jadi", "jdi": "jadi",
    "sm": "sama", "brs": "bersama",
    "bs": "bisa", "bsa": "bisa",
    "sdg": "sedang", "lg": "lagi",
    "hrs": "harus", "wajib": "wajib",
    "jgn": "jangan", "jgnn": "jangan",
    "utk": "untuk", "tuk": "untuk",
    "dr": "dari", "dri": "dari",
    "pd": "pada", "ke": "ke",
    "ny": "nya", "na": "nya",
    "aja": "saja", "aj": "saja",
    "deh": "deh", "dong": "dong",
    "sih": "sih", "nih": "ini",
    "tuh": "itu", "itu": "itu",

    # Singkatan konteks
    "tsb": "tersebut", "svp": "sangat",
    "dll": "dan lain-lain", "dsb": "dan sebagainya",
    "dkk": "dan kawan-kawan", "yg": "yang",
    "stlh": "setelah", "sblm": "sebelum",
    "skrg": "sekarang", "skrang": "sekarang",
    "msh": "masih", "sdh": "sudah",
    "jg": "juga", "juga": "juga",
    "sy": "saya", "aku": "aku",
    "km": "kamu", "kamu": "kamu",
    "mrk": "mereka", "mk": "mereka",
    "kt": "kita", "kita": "kita",

    # Kata sifat informal
    "keren": "keren", "mantap": "mantap",
    "oke": "baik", "ok": "baik",
    "bgus": "bagus", "bagus": "bagus",
    "jelek": "jelek", "bgs": "bagus",

    # Kontraksi umum
    "gimana": "bagaimana", "gmn": "bagaimana",
    "kenapa": "mengapa", "knp": "mengapa",
    "kapan": "kapan", "kpn": "kapan",
    "dimana": "di mana", "dmn": "di mana",
    "siapa": "siapa", "spa": "siapa",
}

# Singkatan formal yang sering muncul di teks Indonesia
SINGKATAN_FORMAL = {
    "yth": "yang terhormat",
    "ttd": "ditandatangani",
    "hlm": "halaman",
    "no": "nomor",
    "tgl": "tanggal",
    "bpk": "bapak",
    "ibu": "ibu",
    "sdr": "saudara",
    "sdri": "saudari",
    "prof": "profesor",
    "dr": "doktor",
    "drs": "doktorandus",
}

# Indikator teks informal untuk register detection
INDIKATOR_INFORMAL = set(KAMUS_ALAY.keys()) | {
    "wkwk", "haha", "hihi", "hehe", "xixi",
    "anjir", "anjay", "asik", "asek",
    "btw", "fyi", "cmiiw", "imo",
    "mantul", "kece", "lebay", "bucin",
    "ngerti", "ngedoin", "ngomong",
}

print("✅ Kamus normalisasi dimuat")


# --- FUNGSI LINGUISTIK INTI ---
def normalize_text(text):
    text = re.sub(r'(.)\1{2,}', r'\1\1', str(text))
    words = text.split()
    normalized = []
    for word in words:
        word_clean = re.sub(r'[^\w]', '', word.lower())
        if word_clean in KAMUS_ALAY:
            replacement = KAMUS_ALAY[word_clean]
            replacement = replacement.capitalize() if word[0].isupper() else replacement
            punct = re.sub(r'\w', '', word)
            normalized.append(replacement + punct)
        else:
            normalized.append(word)
    return re.sub(r'\s+', ' ', ' '.join(normalized)).strip()

def detect_register(text):
    text_lower = text.lower()
    words = set(re.findall(r'\b\w+\b', text_lower))
    informal_count = len(words.intersection(INDIKATOR_INFORMAL))
    has_repetition = bool(re.search(r'(.)\1{2,}', text))
    english_words = {"the", "is", "are", "was", "were", "have", "has", "had", "will", "would", "could", "should", "but", "and", "or", "not", "for", "with"}
    has_codeswitching = bool(words.intersection(english_words))
    informal_signals = sum([informal_count >= 2, has_repetition, has_codeswitching, informal_count >= 1 and len(text.split()) < 20])
    return "[INFORMAL]" if informal_signals >= 2 else "[FORMAL]"

# VOICE_MAP = {"Act": "AKTIF", "Pass": "PASIF", None: ""}
# PREFIKS_AKTIF = {"me", "mem", "men", "meng", "meny", "menge"}
# PREFIKS_PASIF = {"di"}
# PREFIKS_LAIN = {"ber": "STATIF", "ter": "PASIF-SPONTAN", "ke": "PASIF-TIDAK-SENGAJA"}
# SUFIKS_MAP = {"kan": "KAUSATIF", "i": "APLIKATIF", "an": "NOMINALISASI"}

# def get_morphological_function(word_text, lemma, upos, feats):
#     if upos != "VERB": return None
#     word_lower = word_text.lower()
#     if word_lower == lemma.lower() or len(word_lower) <= 3: return None
    
#     functions = []
#     voice = None
#     if feats:
#         for feat in feats.split('|'):
#             if feat.startswith('Voice='): voice = VOICE_MAP.get(feat.split('=')[1], "")
            
#     for pref in PREFIKS_AKTIF:
#         if word_lower.startswith(pref) and len(word_lower) > len(pref) + 2: functions.append("AKTIF"); break
#     for pref in PREFIKS_PASIF:
#         if word_lower.startswith(pref) and len(word_lower) > len(pref) + 2: functions.append("PASIF"); break
#     for pref, label in PREFIKS_LAIN.items():
#         if word_lower.startswith(pref) and len(word_lower) > len(pref) + 2: functions.append(label); break
#     for suf, label in SUFIKS_MAP.items():
#         if word_lower.endswith(suf) and len(word_lower) > len(suf) + 2: functions.append(label)
        
#     if voice and not functions: functions.append(voice)
#     return "-".join(functions) if functions else None

# --- PERBAIKAN STRUKTUR DATA: MENGGUNAKAN LIST TERURUT (LONGEST-FIRST MATCHING) ---
VOICE_MAP = {"Act": "AKTIF", "Pass": "PASIF", None: ""}

# Diubah ke LIST terurut dari yang karakter terpanjang agar tidak salah deteksi pola substring
PREFIKS_AKTIF = ["menge", "meng", "meny", "mem", "men", "me"]
PREFIKS_PASIF = ["di"]

# Prefiks lain dipastikan aman menggunakan list of tuples untuk menjamin urutan iterasi
PREFIKS_LAIN = [
    ("ter", "PASIF-SPONTAN"),
    ("ke", "PASIF-TIDAK-SENGAJA"),
    ("ber", "INTRANSITIF")
]
SUFIKS_MAP = {"kan": "KAUSATIF", "i": "APLIKATIF", "an": "NOMINALISASI"}

def get_morphological_function(word_text, lemma, upos, feats):
    # PERBAIKAN BUG #3: Izinkan VERB, dan izinkan NOUN hanya jika berakhiran 'an' (Nominalisasi)
    if upos not in ["VERB", "NOUN"]: 
        return None
        
    word_lower = word_text.lower()
    if upos == "NOUN" and not word_lower.endswith("an"): 
        return None
        
    if word_lower == lemma.lower() or len(word_lower) <= 3: 
        return None
    
    functions = []
    voice = None
    if feats:
        for feat in feats.split('|'):
            if feat.startswith('Voice='): 
                voice = VOICE_MAP.get(feat.split('=')[1], "")
            
    # PERBAIKAN BUG #1: Iterasi terurut Longest-First Matching
    for pref in PREFIKS_AKTIF:
        if word_lower.startswith(pref) and len(word_lower) > len(pref) + 2: 
            functions.append("AKTIF")
            break # Hentikan iterasi jika prefiks terpanjang sudah cocok
            
    for pref in PREFIKS_PASIF:
        if word_lower.startswith(pref) and len(word_lower) > len(pref) + 2: 
            functions.append("PASIF")
            break
            
    for pref, label in PREFIKS_LAIN:
        if word_lower.startswith(pref) and len(word_lower) > len(pref) + 2: 
            functions.append(label)
            break
            
    for suf, label in SUFIKS_MAP.items():
        if word_lower.endswith(suf) and len(word_lower) > len(suf) + 2: 
            functions.append(label)
        
    # PERBAIKAN BUG #2: Pengakuan Jujur di Paper bahwa ini adalah Rule-Based Hybrid Annotation
    if voice and not functions: 
        functions.append(voice)
        
    return "-".join(functions) if functions else None

def extract_morph_tags(text, max_verbs=5):
    try:
        doc = nlp_id(text[:500])
    except Exception:
        return ""
    
    morph_entries = []
    for sent in doc.sentences:
        for word in sent.words:
            if len(morph_entries) >= max_verbs: break
            func = get_morphological_function(word.text, word.lemma or word.text, word.upos, word.feats)
            if func: morph_entries.append((word.text, func, word.lemma or word.text))
            
    if not morph_entries: return ""
    
    parts = []
    seen_lemmas = set()
    for word, func, lemma in morph_entries:
        if lemma.lower() not in seen_lemmas:
            parts.append(f"{word}({func}:{lemma})")
            seen_lemmas.add(lemma.lower())
    return " | ".join(parts)

⏳ Mengunduh dan memuat model Stanza Indonesia...


2026-07-12 17:11:14 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
2026-07-12 17:11:14 WARNING: Language id package default expects mwt, which has been added
2026-07-12 17:11:14 INFO: Downloading these customized packages for language: id (Indonesian)...
| Processor       | Package      |
----------------------------------
| tokenize        | gsd          |
| mwt             | gsd          |
| pos             | gsd_charlm   |
| lemma           | gsd_nocharlm |
| backward_charlm | oscar2023    |
| forward_charlm  | oscar2023    |
| pretrain        | conll17      |



models/tokenize/gsd.pt: reconstructing file:   0%|          |  0.00B /  659kB            

models/tokenize/gsd.pt: downloading bytes:           |  0.00B            

2026-07-12 17:11:15 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/id/tokenize/gsd.pt


...

2026-07-12 17:11:21 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/id/forward_charlm/oscar2023.pt


models/pretrain/conll17.pt: reconstructing file:   0%|          |  0.00B /  107MB            

models/pretrain/conll17.pt: downloading bytes:           |  0.00B            

2026-07-12 17:11:24 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/id/pretrain/conll17.pt
2026-07-12 17:11:24 INFO: Finished downloading models and saved to /root/.cache/stanza/1.13.0/resources


✅ Stanza siap digunakan!
✅ Kamus normalisasi dimuat


## Cell 5 — Load & Format Datasets (Chat Template Implementation)

In [14]:
print("""
# ============================================================
# Cell 5 — Load & Format Datasets (Chat Template Implementation)
# ============================================================
""")


# ============================================================
# Cell 5 — Load & Format Datasets (Chat Template Implementation)
# ============================================================



In [15]:
# ============================================================
# CELL 5 — CONFIGURATION-DRIVEN PREPROCESSING (METRIK STANDAR)
# ============================================================

from datasets import load_dataset, Dataset, concatenate_datasets
import pandas as pd
from tqdm import tqdm

def build_linguistic_prompt(instruction, output, mode):
    """
    Membangun prompt ChatML berdasarkan tingkat rekayasa fitur teks yang dipilih.
    """
    inst_processed = instruction
    out_processed = output
    user_msg = ""
    
    # 1. Eksekusi Normalisasi Teks jika diaktifkan
    if mode in ["NORMALIZATION_ONLY", "FULL_PIPELINE", "NORM_MORPH"]:
        inst_processed = normalize_text(instruction)
        out_processed = normalize_text(output)
        
    # 2. Ekstraksi penanda komponen tag linguistik
    register_tag = ""
    if mode in ["REGISTER_ONLY", "FULL_PIPELINE"]:
        register_tag = detect_register(instruction) + "\n"
        
    morph_tag = ""
    if mode in ["MORPH_ONLY", "FULL_PIPELINE", "NORM_MORPH"]:
        combined_text = inst_processed + " " + out_processed[:200]
        morph_line = extract_morph_tags(combined_text)
        if morph_line:
            morph_tag = f"[MORPH: {morph_line}]\n"
            
    # 3. Satukan seluruh komponen ke format final ChatML
    user_msg = f"{register_tag}{morph_tag}Instruksi:\n{inst_processed}"

    messages = [
        {"role": "user", "content": user_msg}, 
        {"role": "assistant", "content": out_processed}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


# --- PROSES PEMBENTUKAN KORPUS LATIHAN ---
print(f"⏳ Menyiapkan data latihan menggunakan mode: {CONFIG['linguistic_mode']}...")

raw_bactrian = load_dataset("MBZUAI/Bactrian-X", "id", split="train").shuffle(seed=42).select(range(CONFIG["train_samples"]))
raw_wiki_df = pd.read_csv(CONFIG['wiki_dataset_dir']).sample(n=3500, random_state=42)
raw_nli = load_dataset("afaji/indonli", split="train", trust_remote_code=True).shuffle(seed=42).select(range(7000))

b_processed = []
for row in tqdm(raw_bactrian, desc="Bactrian NLP"):
    inst = (row.get("instruction") or "").strip()
    inp = (row.get("input") or "").strip()
    full_inst = f"{inst}\n{inp}" if inp else inst
    b_processed.append(build_linguistic_prompt(full_inst, (row.get("output") or "").strip(), CONFIG["linguistic_mode"]))
b_ds = Dataset.from_list(b_processed)

w_processed = []
for _, row in tqdm(raw_wiki_df.iterrows(), total=len(raw_wiki_df), desc="Wiki NLP"):
    w_processed.append(build_linguistic_prompt(str(row.get("instruction", "")), str(row.get("output", "")), CONFIG["linguistic_mode"]))
w_ds = Dataset.from_list(w_processed)

# IndoNLI Selalu Polos Tanpa Intervensi Tag untuk Menjaga Keaslian Ujian Logika
n_processed = []
LABEL_MAP = {0: "ENTAILMENT", 1: "NEUTRAL", 2: "CONTRADICTION"}
for row in tqdm(raw_nli, desc="IndoNLI Formatting"):
    p, h = row.get("premise", ""), row.get("hypothesis", "")
    label = LABEL_MAP.get(row.get("label", 1), "NEUTRAL")
    user_msg = (
        "Diberikan sebuah Premis dan Hipotesis. Tentukan hubungan logis di antara keduanya.\n"
        "Jawab HANYA dengan satu kata: ENTAILMENT, NEUTRAL, atau CONTRADICTION.\n\n"
        f"Premis: {p}\n"
        f"Hipotesis: {h}"
    )
    messages = [{"role": "user", "content": user_msg}, {"role": "assistant", "content": label}]
    n_processed.append({"text": tokenizer.apply_chat_template(messages, tokenize=False)})
n_ds = Dataset.from_list(n_processed)

train_dataset = concatenate_datasets([b_ds, n_ds, w_ds]).shuffle(seed=42)
split_data = train_dataset.train_test_split(test_size=500, seed=42)
train_dataset = split_data["train"]
eval_dataset = split_data["test"] # validation

print(f"✅ Selesai! Total Data Training Akhir: {len(train_dataset)} sampel")

⏳ Menyiapkan data latihan menggunakan mode: NORM_MORPH...


README.md:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Bactrian-X.py:   0%|          | 0.00/2.26k [00:00<?, ?B/s]

id/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 29.8MB            

id/train/0000.parquet: downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/2197 [00:00<?, ? examples/s]

Generating test_lay split:   0%|          | 0/2201 [00:00<?, ? examples/s]

Generating test_expert split:   0%|          | 0/2984 [00:00<?, ? examples/s]

IndoNLI Formatting: 100%|██████████| 7000/7000 [00:00<00:00, 9254.97it/s]

✅ Selesai! Total Data Training Akhir: 18000 sampel


In [16]:
# ---- Dataset Evaluasi (Zero-shot / Post-training) ----
print("Loading Dataset Evaluasi Tambahan...")
eval_tasks = {}

try:
    eval_tasks["copal"] = load_dataset("haryoaw/COPAL", split="test", trust_remote_code=True)
    print(f"✅ COPAL-ID: {len(eval_tasks['copal'])} sampel")
except Exception as e:
  print(f"❌ COPAL-ID gagal diload: {e}")

try:
    eval_tasks["smsa"] = load_dataset("indonlp/indonlu", "smsa", split="test", trust_remote_code=True)
    print(f"✅ SmSA: {len(eval_tasks['smsa'])} sampel")
except Exception as e:
  print(f"❌ SmSA gagal diload: {e}")

try:
    eval_tasks["indonli"] = load_dataset("afaji/indonli", split="test_expert", trust_remote_code=True)
    print(f"✅ IndoNLI: {len(eval_tasks['indonli'])} sampel")
except Exception as e:
  print(f"❌ IndoNLI gagal diload: {e}")

try:
    eval_tasks["indoculture"] = load_dataset("indolem/IndoCulture", split="test", trust_remote_code=True)
    print(f"✅ IndoCulture: {len(eval_tasks['indoculture'])} sampel")
except Exception as e:
  print(f"❌ IndoCulture gagal diload: {e}")

try:
    # Menggunakan TyDiQA Gold Passage khusus split bahasa Indonesia
    qa_full = load_dataset("tydiqa", "secondary_task", split="validation")

    # Filter hanya untuk bahasa Indonesia
    eval_tasks["qa"] = qa_full.filter(lambda x: x["id"].startswith("indonesian"))
    print(f"✅ QA (TyDiQA Official): {len(eval_tasks['qa'])} sampel")
except Exception as e:
    print(f"❌ QA gagal diload: {e}")

Loading Dataset Evaluasi Tambahan...


README.md:   0%|          | 0.00/3.24k [00:00<?, ?B/s]

test_copal.csv:   0%|          | 0.00/71.3k [00:00<?, ?B/s]

test_copal_colloquial.csv:   0%|          | 0.00/63.1k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/559 [00:00<?, ? examples/s]

...

test.csv:   0%|          | 0.00/702k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2429 [00:00<?, ? examples/s]

✅ IndoCulture: 2429 sampel


README.md:   0%|          | 0.00/11.1k [00:00<?, ?B/s]

❌ QA gagal diload: Invalid HF URI 'hf://datasets/tydiqa@da78f23f9119363459acbaf46bf89426ff26c259/.huggingface.yaml'. Repository id must be 'namespace/name', got 'tydiqa'.


In [17]:
# ====================================================================
# SEL BARU: ANALISIS STATISTIK DISTRIBUSI FITUR MORFOLOGI (SENJATA SIDANG)
# ====================================================================
from collections import Counter
import pandas as pd
from tqdm import tqdm

print("⏳ Menghitung distribusi kemunculan tag morfologi pada training set...")
all_tags_extracted = []
sample_illustrations = []

# Scan cepat ke dataset latih asli untuk melihat hasil ekstraksi teratur
for row in tqdm(raw_bactrian, desc="Scanning Bactrian-X"):
    text = (row.get("instruction") or "") + " " + (row.get("input") or "")
    try:
        doc = nlp_id(text[:500])
        for sent in doc.sentences:
            for word in sent.words:
                func = get_morphological_function(word.text, word.lemma or word.text, word.upos, word.feats)
                if func:
                    all_tags_extracted.append(func)
                    if len(sample_illustrations) < 15:
                        sample_illustrations.append({
                            "Kata Asli": word.text,
                            "Fungsi Semantik (Tag)": func,
                            "Kata Dasar (Lemma)": word.lemma or word.text
                        })
    except Exception:
        continue

# 1. TABEL UTAMA UNTUK BAB 4 TESIS / PAPER
print("\n" + "="*50)
print("📊 RESEARCH ISSUE #1: TABEL FREKUENSI TAG MORFOLOGI")
print("="*50)
tag_counts = Counter(all_tags_extracted)
df_tags = pd.DataFrame(tag_counts.most_common(), columns=["Jenis Tag Morfologi", "Jumlah Kemunculan (Count)"])
df_tags["Persentase (%)"] = round((df_tags["Jumlah Kemunculan (Count)"] / len(all_tags_extracted)) * 100, 2)
print(df_tags.to_string(index=False))

# 2. TABEL CONTOH STRUKTUR UNTUK LITERATUR BAB 3
print("\n" + "="*50)
print("📝 RESEARCH ISSUE #2: CONTOH KATA DAN PREDIKSI FITUR")
print("="*50)
df_samples = pd.DataFrame(sample_illustrations)
print(df_samples.drop_duplicates(subset=["Fungsi Semantik (Tag)"]).to_string(index=False))

print("\n📊 Rata-rata tag morfologi per sampel instruksi: ", round(len(all_tags_extracted)/len(raw_bactrian), 2))

⏳ Menghitung distribusi kemunculan tag morfologi pada training set...


Scanning Bactrian-X: 100%|██████████| 8000/8000 [06:40<00:00, 19.98it/s]


📊 RESEARCH ISSUE #1: TABEL FREKUENSI TAG MORFOLOGI
                      Jenis Tag Morfologi  Jumlah Kemunculan (Count)  Persentase (%)
                             NOMINALISASI                       4830           23.48
                    KAUSATIF-NOMINALISASI                       2947           14.33
              AKTIF-KAUSATIF-NOMINALISASI                       2396           11.65
                                    AKTIF                       1937            9.42
              PASIF-KAUSATIF-NOMINALISASI                       1837            8.93
                          AKTIF-APLIKATIF                       1648            8.01
         PASIF-TIDAK-SENGAJA-NOMINALISASI                       1253            6.09
                              INTRANSITIF                       1096            5.33
                                    PASIF                        659            3.20
                          PASIF-APLIKATIF                        517            2.51
             

In [18]:
# # ====================================================================
# # SEL SIMULASI: INTERACTIVE LINGUISTIC PREPROCESSING CHECKER
# # ====================================================================

# # 1. Siapkan beberapa contoh kalimat uji coba (informal internet & formal kompleks)
# test_sentences = [
#     {
#         "instruction": "gw blm makan nih krn udh males bgt keluar, ntar ketabrak motor lg wkwk",
#         "output": "oke nanti aku belikan makanan lewat ojek online ya."
#     },
#     {
#         "instruction": "Pemerintah sedang memperluas wilayah konservasi budaya untuk mempertahankan tradisi leluhur.",
#         "output": "Kebijakan tersebut diambil demi menjaga kelestarian identitas bangsa."
#     },
#     {
#         "instruction": "lo tau gak sih kenapa dia menangis semalam? katanya kepleset di kamar mandi.",
#         "output": "Kelepasan bicara membuat dia merasa bersalah kepada ibunya."
#     }
# ]

# print("="*75)
# print("🔍 LIVE PREPROCESSING DIAGNOSTIC DASHBOARD")
# print("="*75)

# for idx, sample in enumerate(test_sentences, 1):
#     inst = sample["instruction"]
#     out = sample["output"]
    
#     # Eksekusi fungsi satu per satu untuk melihat efek tiap komponen
#     text_normalized = normalize_text(inst)
#     register_detected = detect_register(inst)
    
#     # Gabungkan instruksi dan sedikit output untuk mencari kata kerja verba (sesuai logika asli C3)
#     combined_for_morph = text_normalized + " " + normalize_text(out)[:200]
#     morph_tags = extract_morph_tags(combined_for_morph)
    
#     # Cetak hasil pembongkaran fitur teks
#     print(f"\n📌 [CONTOH KALIMAT #{idx}]")
#     print(f"   • Teks Asli   : \"{inst}\"")
#     print(f"   • Normalisasi : \"{text_normalized}\"")
#     print(f"   • Register Tag: {register_detected}")
#     print(f"   • Morph Tag   : {f'[MORPH: {morph_tags}]' if morph_tags else 'Tidak ditemukan verba berimbuhan'}")
    
#     print(f"\n   ⚙️ [HASIL AKHIR PROMPT CHATML - MODE FULL_PIPELINE]:")
#     # Memanggil fungsi pembangun prompt utama Anda
#     final_prompt = build_linguistic_prompt(inst, out, mode="FULL_PIPELINE")
#     print("-" * 50)
#     print(final_prompt["text"])
#     print("-" * 50)
#     print("="*75)

In [19]:
# ====================================================================
# INTERACTIVE LINGUISTIC PREPROCESSING CHECKER (CORRECTED)
# Morphological analysis is performed ONLY on the instruction
# to avoid target leakage.
# ====================================================================

test_sentences = [
    {
        "instruction": "gw blm makan nih krn udh males bgt keluar, ntar ketabrak motor lg wkwk",
        "output": "oke nanti aku belikan makanan lewat ojek online ya."
    },
    {
        "instruction": "Pemerintah sedang memperluas wilayah konservasi budaya untuk mempertahankan tradisi leluhur.",
        "output": "Kebijakan tersebut diambil demi menjaga kelestarian identitas bangsa."
    },
    {
        "instruction": "lo tau gak sih kenapa dia menangis semalam? katanya kepleset di kamar mandi.",
        "output": "Kelepasan bicara membuat dia merasa bersalah kepada ibunya."
    }
]

print("=" * 75)
print("🔍 LIVE PREPROCESSING DIAGNOSTIC DASHBOARD")
print("=" * 75)

for idx, sample in enumerate(test_sentences, 1):

    inst = sample["instruction"]
    out = sample["output"]

    # ============================================================
    # Step 1. Normalize ONLY the instruction
    # ============================================================
    text_normalized = normalize_text(inst)

    # ============================================================
    # Step 2. Detect register from ONLY the instruction
    # ============================================================
    register_detected = detect_register(inst)

    # ============================================================
    # Step 3. Morphological analysis ONLY on the instruction
    # (No response information is allowed here)
    # ============================================================
    morph_tags = extract_morph_tags(text_normalized)

    print(f"\n📌 [CONTOH KALIMAT #{idx}]")
    print(f"   • Teks Asli   : \"{inst}\"")
    print(f"   • Normalisasi : \"{text_normalized}\"")
    print(f"   • Register Tag: {register_detected}")

    if morph_tags:
        print(f"   • Morph Tag   : [MORPH: {morph_tags}]")
    else:
        print("   • Morph Tag   : Tidak ditemukan fitur morfologi")

    print("\n   ⚙️ [HASIL AKHIR PROMPT CHATML - MODE FULL_PIPELINE]:")

    final_prompt = build_linguistic_prompt(
        instruction=inst,
        output=out,
        mode="FULL_PIPELINE"
    )

    print("-" * 50)
    print(final_prompt["text"])
    print("-" * 50)

print("=" * 75)

🔍 LIVE PREPROCESSING DIAGNOSTIC DASHBOARD

📌 [CONTOH KALIMAT #1]
   • Teks Asli   : "gw blm makan nih krn udh males bgt keluar, ntar ketabrak motor lg wkwk"
   • Normalisasi : "saya belum makan ini karena sudah males banget keluar, ntar ketabrak motor lagi wkwk"
   • Register Tag: [INFORMAL]
   • Morph Tag   : Tidak ditemukan fitur morfologi

   ⚙️ [HASIL AKHIR PROMPT CHATML - MODE FULL_PIPELINE]:
--------------------------------------------------
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
[INFORMAL]
[MORPH: belikan(KAUSATIF-NOMINALISASI:belik) | makanan(NOMINALISASI:makan)]
Instruksi:
saya belum makan ini karena sudah males banget keluar, ntar ketabrak motor lagi wkwk<|im_end|>
<|im_start|>assistant
baik nanti aku belikan makanan lewat ojek online ya.<|im_end|>

--------------------------------------------------

📌 [CONTOH KALIMAT #2]
   • Teks Asli   : "Pemerintah sedang memperluas wilayah konservasi budaya untuk mempertahankan tradisi leluhur."
   • No

## Cell 6 — Batched Evaluation & Efficiency Metrics

In [20]:
print("""
# ============================================================
# Cell 6 — Batched Evaluation & Efficiency Metrics
# ============================================================
""")


# ============================================================
# Cell 6 — Batched Evaluation & Efficiency Metrics
# ============================================================



In [21]:
import collections
import re
import string

In [22]:
# ==============================================================================
# CELL 6 — DYNAMICAL BATCHED EVALUATION (METRIK STANDAR ACCURACY & F1)
# ==============================================================================

import collections
import re
import string
import time
import torch
from tqdm import tqdm

# --- 1. UTALITAS NORMALISASI DAN SKORING QA (TYDIQA) ---
def normalize_qa_answer(s):
    """Membersihkan teks dari tanda baca, spasi berlebih, dan mengubah ke huruf kecil."""
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def white_space_fix(text):
        return ' '.join(text.split())
    def lower(text):
        return text.lower()
    return white_space_fix(remove_punc(lower(str(s))))

def qa_exact_match(prediction, ground_truth):
    return 1.0 if normalize_qa_answer(prediction) == normalize_qa_answer(ground_truth) else 0.0

def qa_f1_score(prediction, ground_truth):
    pred_tokens = normalize_qa_answer(prediction).split()
    truth_tokens = normalize_qa_answer(ground_truth).split()

    if not pred_tokens and not truth_tokens: return 1.0
    if not pred_tokens or not truth_tokens: return 0.0

    common = collections.Counter(pred_tokens) & collections.Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0: return 0.0

    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(truth_tokens)
    return (2 * precision * recall) / (precision + recall)


# --- 2. MESIN INFERENSI BATCH DENGAN SAKLAR OTOMATIS ---
def evaluate_batched_final(model, tokenizer, dataset, task_name, linguistic_mode, batch_size=8):
    """
    Fungsi evaluasi tunggal yang otomatis menyusun prompt berdasarkan saklar
    CONFIG['linguistic_mode'] yang sedang aktif saat ini.
    """
    model.eval()
    results = []

    # Penentuan tag register otomatis sejalan dengan training pipeline
    tag_prefix = ""
    if linguistic_mode in ["REGISTER_ONLY", "FULL_PIPELINE"]:
        tag_prefix = "[INFORMAL]\n" if task_name == "smsa" else "[FORMAL]\n"

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Eval {task_name}"):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        prompts = []

        for s in batch:
            # A. KONDISI KHUSUS INDONLI (Selalu polos tanpa kata 'Instruksi' sesuai training)
            if task_name == "indonli":
                p = (
                    "Diberikan sebuah Premis dan Hipotesis. Tentukan hubungan logis di antara keduanya.\n"
                    "Jawab HANYA dengan satu kata: ENTAILMENT, NEUTRAL, atau CONTRADICTION.\n\n"
                    f"Premis: {s['premise']}\n"
                    f"Hipotesis: {s['hypothesis']}"
                )
            
            # B. KONDISI JALUR COPAL
            elif task_name == "copal":
                q_type = str(s.get("question", "")).lower()
                pertanyaan = "Apa penyebab dari situasi tersebut?" if q_type == "cause" else \
                             "Apa akibat dari situasi tersebut?" if q_type == "effect" else \
                             "Manakah pilihan yang paling masuk akal?"
                p = (
                    f"{tag_prefix}Instruksi:\n"
                    f"Situasi: {s['premise']}\n"
                    f"Pertanyaan: {pertanyaan}\n"
                    f"A. {s['choice1']}\n"
                    f"B. {s['choice2']}\n"
                    "Jawaban (A/B):"
                )

            # C. KONDISI JALUR QA (TYDIQA)
            elif task_name == "qa":
                passage_text = " ".join(s.get("passage", [])).replace(" ,", ",").replace(" .", ".")
                question_text = " ".join(s.get("question", [])).replace(" ,", ",").replace(" ?", "?")
                p = (
                    f"{tag_prefix}Instruksi:\n"
                    "Berdasarkan teks berikut, jawablah pertanyaan dengan singkat dan tepat.\n\n"
                    f"Teks: {passage_text}\n"
                    f"Pertanyaan: {question_text}\n\n"
                    "Jawaban:"
                )

            # D. KONDISI JALUR INDOCULTURE
            elif task_name == "indoculture":
                options_list = s.get("options", [])
                options_text = "\n".join(options_list) if isinstance(options_list, list) else str(options_list)
                p = (
                    f"{tag_prefix}Instruksi:\n"
                    "Bacalah pertanyaan berikut and pilih jawaban yang paling tepat berdasarkan konteks budaya Indonesia.\n\n"
                    f"Konteks: {s['context']}\n"
                    f"Pilihan:\n{options_text}\n\n"
                    "Jawaban (A, B, atau C):"
                )

            # E. KONDISI JALUR SMSA
            else:
                p = (
                    f"{tag_prefix}Instruksi:\n"
                    "Analisis sentimen dari teks berikut. Jawab HANYA dengan satu kata: POSITIF, NETRAL, atau NEGATIF.\n\n"
                    f"Teks: {s['text']}\n"
                    "Jawaban:"
                )

            # Bungkus teks prompt ke standar ChatML Qwen
            msg = [{"role": "user", "content": p}]
            prompts.append(tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True))

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            start_inf = time.perf_counter()
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
            end_inf = time.perf_counter()

        for j, out in enumerate(outputs):
            gen_text = tokenizer.decode(out[inputs["input_ids"].shape[1]:], skip_special_tokens=True).upper()
            results.append({"pred": gen_text, "latency": (end_inf - start_inf)/len(batch)})

    return results


# --- 3. EKSTRAKSI JAWABAN MODEL SECARA PEMAAF (HYBRID EXTRACTOR) ---
def extract_hybrid_prediction(task_name, gen_text):
    """
    Mengekstrak label jawaban secara pemaaf menggunakan Regex,
    mengakomodasi model finetuned yang mengalami Excessive Verbosity.
    """
    gen_text = str(gen_text).strip().upper()

    if task_name in ["copal", "indoculture"]:
        match = re.search(r'^([A-C])[\.\s\(\:]', gen_text)
        if match: return match.group(1)
        if len(gen_text) == 1 and gen_text in ['A', 'B', 'C']: return gen_text
        for label in ["A", "B", "C"]:
            if f"JAWABAN: {label}" in gen_text or f"ADALAH {label}" in gen_text or f"PILIHAN {label}" in gen_text:
                return label

    elif task_name == "indonli":
        if "SESUAI" in gen_text or "ENTAILMENT" in gen_text: return "SESUAI"
        if "BERTENTANGAN" in gen_text or "CONTRADICTION" in gen_text: return "BERTENTANGAN"
        if "NETRAL" in gen_text or "NEUTRAL" in gen_text: return "NETRAL"

    elif task_name == "smsa":
        if "POSITIF" in gen_text or "POSITIVE" in gen_text: return "POSITIF"
        if "NEGATIF" in gen_text or "NEGATIVE" in gen_text: return "NEGATIF"
        if "NETRAL" in gen_text or "NEUTRAL" in gen_text: return "NETRAL"

    elif task_name == "qa":
        clean_text = gen_text.split('\n')[0].strip()
        match = re.search(r'^[A-C]\s*[\(\.\:]\s*(.*)', clean_text)
        if match: return match.group(1).replace(")", "").strip()
        return clean_text

    return "UNKNOWN"


# --- 4. PENYELARASAN LABEL TARGET DATA (GROUND TRUTH MAPPER) ---
def map_hybrid_label(task_name, sample):
    """Menyamakan format label dataset (Mengatasi tipe data string/integer yang bentrok)"""
    if task_name == "qa":
       passage_tokens = sample.get("passage", [])
       labels = sample.get("seq_label", [])
       ans_tokens = [tok for tok, lbl in zip(passage_tokens, labels) if lbl in [1, 2]]
       true_answer = " ".join(ans_tokens).strip().upper()
       return true_answer.replace(" ,", ",").replace(" .", ".") if true_answer else "UNKNOWN"

    raw_label = str(sample.get("label", sample.get("answer", sample.get("target", "-1")))).strip().upper()

    if task_name == "copal":
        return "A" if raw_label == "0" else "B"
    elif task_name == "indoculture":
        label_map = {"0": "A", "1": "B", "2": "C"}
        return raw_label if raw_label in ["A", "B", "C"] else label_map.get(raw_label, "UNKNOWN")
    elif task_name == "indonli":
        label_map = {"0": "SESUAI", "1": "NETRAL", "2": "BERTENTANGAN"}
        return label_map.get(raw_label, "UNKNOWN")
    elif task_name == "smsa":
        label_map = {"0": "POSITIF", "1": "NETRAL", "2": "NEGATIF"}
        return raw_label if raw_label in ["POSITIF", "NETRAL", "NEGATIF"] else label_map.get(raw_label, "UNKNOWN")

    return raw_label

## Cell 7 — Eksperimen 1: Baseline (Zero-Shot)

In [23]:
# print("""
# # ============================================================
# # Cell 7 — Eksperimen 1: Baseline (Zero-Shot)
# # ============================================================
# """)

In [24]:
import json
from sklearn.metrics import accuracy_score, f1_score

In [25]:
# # ============================================================
# # CELL 7 — Eksperimen 1: Baseline (Zero-Shot)
# # ============================================================
# print("=" * 60)
# print("EVALUASI ZERO-SHOT (BASELINE) - SEBELUM QLORA")
# print("=" * 60)

# baseline_results = {}

# # # --- FUNGSI MAPPING GROUND TRUTH ---
# # def map_hybrid_label(task, sample):
# #     if task == "indoculture":
# #         return str(sample.get("answer", "UNKNOWN")).strip().upper()
# #     elif task == "qa": # Untuk FacQA
# #         # Jawaban QA biasanya ada di kolom 'answer' atau list 'answers'
# #         ans = sample.get("answer", sample.get("answers", ["UNKNOWN"]))
# #         if isinstance(ans, list) and len(ans) > 0: return str(ans[0]).strip().upper()
# #         return str(ans).strip().upper()

# #     raw_label = str(sample.get("label", "-1"))
# #     if task == "copal":
# #         return "A" if raw_label == "0" else "B"
# #     elif task == "smsa":
# #         return {"0": "POSITIF", "1": "NETRAL", "2": "NEGATIF"}.get(raw_label, "UNKNOWN")
# #     elif task == "indonli":
# #         return {"0": "ENTAILMENT", "1": "NEUTRAL", "2": "CONTRADICTION"}.get(raw_label, "UNKNOWN")
# #     return raw_label

# # # --- FUNGSI EKSTRAKSI "PEMAAF" (DARI NOTEBOOK LAMA) ---
# # def extract_hybrid_prediction(task, pred_text):
# #     text = pred_text.strip().lower()

# #     if task in ["copal", "indoculture"]:
# #         if "jawaban: a" in text or "pilihan a" in text: return "A"
# #         if "jawaban: b" in text or "pilihan b" in text: return "B"
# #         if "jawaban: c" in text or "pilihan c" in text: return "C"
# #         # Jika menjawab panjang, ambil karakter alfabet pertama yang muncul
# #         chars = [c for c in text if c.isalpha()]
# #         if chars and chars[0] in ['a', 'b', 'c']: return chars[0].upper()

# #     elif task == "smsa":
# #         if "positif" in text or "positive" in text: return "POSITIF"
# #         if "negatif" in text or "negative" in text: return "NEGATIF"
# #         if "netral" in text or "neutral" in text: return "NETRAL"

# #     elif task == "indonli":
# #         if "entailment" in text: return "ENTAILMENT"
# #         if "contradiction" in text: return "CONTRADICTION"
# #         if "neutral" in text or "netral" in text: return "NEUTRAL"

# #     elif task == "qa":
# #         # Ambil kalimat pertama saja sebagai tebakan ekstraksi QA
# #         return pred_text.strip().split('\n')[0].strip().upper()

# #     return "UNKNOWN"

# # --- LOOP EVALUASI ---
# for task_name, dataset in eval_tasks.items():
#     if dataset is None: continue

#     print(f"\n🚀 Mengevaluasi {task_name.upper()}...")

#     # Ambil subset agar hemat waktu (Sesuai CONFIG["eval_limit"])
#     eval_subset = dataset.select(range(min(CONFIG["eval_limit"], len(dataset))))

#     # Pastikan di Cell 6 nama fungsinya adalah evaluate_batched
#     raw_results = evaluate_batched(model, tokenizer, eval_subset, task_name=task_name, batch_size=CONFIG["batch_size"])

#     preds, labels, latencies = [], [], []
#     unknown_count = 0

#     for i, res in enumerate(raw_results):
#         # 1. Ekstraksi Prediksi
#         pred_final = extract_hybrid_prediction(task_name, res["pred"])
#         # 2. Ambil Label Asli
#         label_final = map_hybrid_label(task_name, eval_subset[i])

#         preds.append(pred_final)
#         labels.append(label_final)
#         latencies.append(res["latency"])

#         if pred_final == "UNKNOWN":
#             unknown_count += 1

#     # --- PERHITUNGAN JUJUR (Honest Metrics) ---
#     # Membandingkan seluruh data, UNKNOWN akan otomatis disalahkan
#     # Untuk QA, kita pakai exact match sederhana sebagai Accuracy
#     acc = accuracy_score(labels, preds) * 100

#     # F1 Score hanya untuk klasifikasi, QA dilewati
#     if task_name == "qa":
#         # Gunakan metrik khusus QA (EM dan Token-F1)
#         em_scores = [qa_exact_match(p, l) for p, l in zip(preds, labels)]
#         f1_scores = [qa_f1_score(p, l) for p, l in zip(preds, labels)]
        
#         acc = np.mean(em_scores) * 100      # Exact Match diwakili sebagai Akurasi
#         mac_f1 = np.mean(f1_scores) * 100   # Token-F1 diwakili sebagai Macro F1
#     else:
#         # Gunakan metrik klasifikasi standar untuk COPAL, SmSA, IndoNLI, dll
#         acc = accuracy_score(labels, preds) * 100
#         mac_f1 = f1_score(labels, preds, average="macro", zero_division=0) * 100
    
#     avg_latency = np.mean(latencies)

#     print(f"    📊 Accuracy (Total) : {acc:.2f}%")
#     if task_name != "qa": print(f"    📊 Macro F1         : {mac_f1:.2f}%")
#     print(f"    ⏱️ Avg Latency      : {avg_latency:.4f} s/sampel")
#     print(f"    ⚠️ Format Gagal     : {unknown_count} dari {len(preds)} sampel")

#     baseline_results[task_name] = {"accuracy": acc, "macro_f1": mac_f1, "latency": avg_latency, "fail": unknown_count}

# # Simpan hasil
# with open("/kaggle/working/Riset_QLoRA/baseline_results.json", "w") as f:
#     json.dump(baseline_results, f, indent=4)
# print("\n✅ Evaluasi Zero-Shot Selesai & Tersimpan!")

# # Simpan hasil langsung ke Google Drive
# # with open("/content/drive/MyDrive/Riset_QLoRA/baseline_results.json", "w") as f:
# #     json.dump(baseline_results, f, indent=4)
# # print("\n✅ Evaluasi Zero-Shot Selesai & Tersimpan!")

## Cell C4 - Vocabulary Adaptation/Extendtion (OLD)

In [26]:
# print("="*60)
# print("TAHAP C4: VOCABULARY ADAPTATION (DATA-DRIVEN)")
# print("="*60)

In [27]:
# from transformers import TrainerCallback

In [28]:
# # 1. AMBIL TEKS DARI DATASET C3 (Huruf kecil untuk konsistensi)
# all_texts = []
# for sample in train_dataset:
#     if "text" in sample: all_texts.append(str(sample["text"]).lower())

# # 2. HITUNG FREKUENSI KATA ABSOLUT
# word_freq = Counter()
# for text in all_texts:
#     # Hanya ambil kata alfabetik minimal 4 huruf (membantu memfilter angka/noise)
#     words = re.findall(r'\b[a-z]{4,}\b', text)
#     word_freq.update(words)

# # ====================================================================
# # 3. SELEKSI BERDASARKAN FERTILITY & FREQUENCY THRESHOLD (SCIENTIFIC)
# # ====================================================================
# MIN_FREQ = 15          # Kata harus muncul minimal 15 kali di corpus agar layak dipelajari
# FERTILITY_THRESH = 3   # Kata harus dipecah menjadi minimal 3 subword oleh tokenizer asli

# tokens_to_add = []
# fertility_stats = []

# for word, freq in word_freq.items():
#     if freq >= MIN_FREQ:
#         # Hitung rasio Fertility: Jumlah subword dibagi 1 kata asli
#         n_tokens = len(tokenizer.encode(word, add_special_tokens=False))
        
#         if n_tokens >= FERTILITY_THRESH:
#             tokens_to_add.append(word)
#             fertility_stats.append({"word": word, "freq": freq, "fertility": n_tokens})

# print(f"✅ Ambang Batas: Frekuensi >= {MIN_FREQ}, Fertility >= {FERTILITY_THRESH}")
# print(f"✅ Ditemukan {len(tokens_to_add)} token valid yang lulus seleksi ilmiah.")

# # Tampilkan beberapa contoh untuk Sanity Check
# fertility_stats.sort(key=lambda x: x["freq"], reverse=True) # Sortir HANYA untuk tampilan, bukan seleksi
# print("\n🔍 10 Kandidat Teratas (Sanity Check):")
# for stat in fertility_stats[:10]:
#     print(f"   - {stat['word']:<15} | Freq: {stat['freq']:<5} | Fertility: {stat['fertility']}")

# # 4. EXTEND TOKENIZER & RESIZE EMBEDDING
# vocab_size_before = len(tokenizer)
# tokenizer.add_tokens(tokens_to_add)
# model.resize_token_embeddings(len(tokenizer))

# # 5. INIT DENGAN SUBWORD AVERAGING
# embedding_matrix = model.get_input_embeddings().weight
# with torch.no_grad():
#     for token in tokens_to_add:
#         new_id = tokenizer.convert_tokens_to_ids(token)
#         subword_ids = tokenizer.encode(token, add_special_tokens=False) 
#         if subword_ids and new_id >= vocab_size_before:
#             embedding_matrix[new_id] = embedding_matrix[subword_ids].mean(dim=0)
            
# print("\n✅ Tokenizer berhasil diperluas dan diinisialisasi dengan Subword Averaging!")

In [29]:
# # 6. SIAPKAN CUSTOM CALLBACK UNTUK TRAINING (Tetap Menggunakan OOP)
# new_token_ids = [tokenizer.convert_tokens_to_ids(t) for t in tokens_to_add if tokenizer.convert_tokens_to_ids(t) is not None]

In [30]:
# class SelectiveEmbeddingCallback(TrainerCallback):
#     """
#     Callback untuk memastikan hanya embedding token baru yang diupdate.
#     Setelah setiap backward pass, zero-out gradient token lama.
#     """
#     def __init__(self, new_token_ids, vocab_size_before):
#         self.new_token_ids = new_token_ids
#         self.vocab_size_before = vocab_size_before

#     def on_step_end(self, args, state, control, model=None, **kwargs):
#         if model is not None:
#             embedding_weight = model.get_input_embeddings().weight
#             if embedding_weight.grad is not None:
#                 mask = torch.zeros(embedding_weight.grad.shape[0], dtype=torch.bool)
#                 mask[self.new_token_ids] = True  
#                 embedding_weight.grad[~mask] = 0 
#         return control

# selective_callback = SelectiveEmbeddingCallback(new_token_ids, vocab_size_before)

## Cell C4 - Vocabulary Adaptation/Extendtion (NEW)

In [31]:
# ====================================================================
# TAHAP C4: VOCABULARY ADAPTATION (DATA-DRIVEN) - STANDALONE CELL
# ====================================================================
from transformers import TrainerCallback
import torch
from collections import Counter
import re

# Inisialisasi default agar tidak memicu NameError di cell bawah jika C4 tidak aktif
selective_callback = None
vocab_size_before = len(tokenizer)

if CONFIG["vocab_adaptation"]:
    print("\n" + "="*60)
    print("🧬 [ENGINE C4] MENJALANKAN ADAPTASI KOSAKATA SECARA ILMIAH...")
    print("="*60)

    # 1. AMBIL TEKS DARI DATASET C3 (Huruf kecil untuk konsistensi)
    all_texts = []
    for sample in train_dataset:
        if "text" in sample: 
            all_texts.append(str(sample["text"]).lower())

    # 2. HITUNG FREKUENSI KATA ABSOLUT
    word_freq = Counter()
    for text in all_texts:
        # Hanya ambil kata alfabetik minimal 4 huruf (membantu memfilter angka/noise)
        words = re.findall(r'\b[a-z]{4,}\b', text)
        word_freq.update(words)

    # ====================================================================
    # 3. SELEKSI BERDASARKAN FERTILITY & FREQUENCY THRESHOLD (SCIENTIFIC)
    # ====================================================================
    MIN_FREQ = 15          # Kata harus muncul minimal 15 kali di corpus agar layak dipelajari
    FERTILITY_THRESH = 3   # Kata harus dipecah menjadi minimal 3 subword oleh tokenizer asli

    tokens_to_add = []
    fertility_stats = []

    for word, freq in word_freq.items():
        if freq >= MIN_FREQ:
            # Hitung rasio Fertility: Jumlah subword dibagi 1 kata asli
            n_tokens = len(tokenizer.encode(word, add_special_tokens=False))
            
            if n_tokens >= FERTILITY_THRESH:
                tokens_to_add.append(word)
                fertility_stats.append({"word": word, "freq": freq, "fertility": n_tokens})

    print(f"   ✅ Ambang Batas: Frekuensi >= {MIN_FREQ}, Fertility >= {FERTILITY_THRESH}")
    print(f"   ✅ Ditemukan {len(tokens_to_add)} token valid yang lulus seleksi ilmiah.")

    # Tampilkan beberapa contoh untuk Sanity Check
    fertility_stats.sort(key=lambda x: x["freq"], reverse=True) # Sortir HANYA untuk tampilan, bukan seleksi
    print("\n🔍 10 Kandidat Teratas (Sanity Check):")
    for stat in fertility_stats[:10]:
        print(f"       - {stat['word']:<15} | Freq: {stat['freq']:<5} | Fertility: {stat['fertility']}")

    # 4. EXTEND TOKENIZER & RESIZE EMBEDDING
    tokenizer.add_tokens(tokens_to_add)
    model.resize_token_embeddings(len(tokenizer))

    # 5. INIT DENGAN SUBWORD AVERAGING
    embedding_matrix = model.get_input_embeddings().weight
    with torch.no_grad():
        for token in tokens_to_add:
            new_id = tokenizer.convert_tokens_to_ids(token)
            subword_ids = tokenizer.encode(token, add_special_tokens=False) 
            if subword_ids and new_id >= vocab_size_before:
                embedding_matrix[new_id] = embedding_matrix[subword_ids].mean(dim=0)
                
    print("\n   ✅ Tokenizer berhasil diperluas dan diinisialisasi dengan Subword Averaging!")

    # 6. SIAPKAN CUSTOM CALLBACK UNTUK TRAINING (Tetap Menggunakan OOP)
    new_token_ids = [tokenizer.convert_tokens_to_ids(t) for t in tokens_to_add if tokenizer.convert_tokens_to_ids(t) is not None]

    class SelectiveEmbeddingCallback(TrainerCallback):
        """
        Callback untuk memastikan hanya embedding token baru yang diupdate.
        Setelah setiap backward pass, zero-out gradient token lama.
        """
        def __init__(self, new_token_ids, vocab_size_before):
            self.new_token_ids = new_token_ids
            self.vocab_size_before = vocab_size_before

        def on_step_end(self, args, state, control, model=None, **kwargs):
            if model is not None:
                embedding_weight = model.get_input_embeddings().weight
                if embedding_weight.grad is not None:
                    mask = torch.zeros(embedding_weight.grad.shape[0], dtype=torch.bool)
                    mask[self.new_token_ids] = True  
                    embedding_weight.grad[~mask] = 0 
            return control

    selective_callback = SelectiveEmbeddingCallback(new_token_ids, vocab_size_before)
else:
    print("\n⏩ Konfigurasi: Langkah Perluasan Kosakata (C4) dinonaktifkan via CONFIG.")


⏩ Konfigurasi: Langkah Perluasan Kosakata (C4) dinonaktifkan via CONFIG.


## Cell 8 — Eksperimen 2: QLoRA Only Training

In [32]:
print("""
# ============================================================
# Cell 8 — Eksperimen 2: QLoRA Only Training
# ============================================================
""")


# ============================================================
# Cell 8 — Eksperimen 2: QLoRA Only Training
# ============================================================



In [33]:
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from trl import SFTTrainer, SFTConfig

# model = prepare_model_for_kbit_training(model)

# # 1. HACK: Ubah DNA asli model agar PEFT tidak tertipu!
# model.config.torch_dtype = torch.float16

# peft_config = LoraConfig(
#     r=CONFIG["lora_r"],
#     lora_alpha=CONFIG["lora_alpha"],
#     target_modules="all-linear",
#     lora_dropout=0.05,
#     task_type="CAUSAL_LM",
# )
# model = get_peft_model(model, peft_config)


# # =========================================================================
# # 🔥 JURUS SAPU BERSIH TOTAL: Musnahkan sisa-sisa BFloat16 sampai ke akar 🔥
# # =========================================================================
# # a. Razia semua Parameter (Bobot/Weights)
# for name, param in model.named_parameters():
#     if param.dtype == torch.bfloat16:
#         param.data = param.data.to(torch.float16)

# # b. Razia semua Buffer (Positional Embeddings, dll) -> INI SARANG ASLINYA
# for name, buffer in model.named_buffers():
#     if buffer.dtype == torch.bfloat16:
#         buffer.data = buffer.data.to(torch.float16)
# # =========================================================================


# training_args = SFTConfig(
#     output_dir=CONFIG["output_dir"],
#     num_train_epochs=CONFIG["epochs"],
#     per_device_train_batch_size=CONFIG["batch_size"],
#     gradient_accumulation_steps=CONFIG["grad_acc_steps"],
#     learning_rate=CONFIG["lr"],
#     logging_steps=20,
#     fp16=False,     # <--- WAJIB: T4 menggunakan fp16, bukan bf16
#     bf16=False,
#     optim="paged_adamw_8bit", # Menghemat VRAM secara signifikan
#     max_length=CONFIG["max_seq_length"],
#     eval_strategy="steps",   # Aktifkan agar eval_dataset benar-benar digunakan
#     eval_steps=100,          # Evaluasi tiap 100 steps
#     save_total_limit=2,      # Jaga agar disk Kaggle tidak penuh
#     save_strategy="steps",   # Simpan setiap 100 steps
#     save_steps=100,
#     report_to="none"
# )

# tokenizer.padding_side = "right"

# trainer = SFTTrainer(
#     model=model,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     args=training_args,
#     # peft_config=peft_config,
#     processing_class=tokenizer,
#     callbacks=[selective_callback] # Tahap C4

# )

# model.get_input_embeddings().weight.requires_grad = True

# print("🚀 Memulai Training QLoRA Only...")
# trainer.train(resume_from_checkpoint=None)

# # Simpan adapter & cek ukuran file
# trainer.save_model(CONFIG["output_dir"])
# adapter_size = sum(os.path.getsize(os.path.join(CONFIG["output_dir"], f)) for f in os.listdir(CONFIG["output_dir"]) if os.path.isfile(os.path.join(CONFIG["output_dir"], f))) / (1024*1024)
# print(f"✅ Training Selesai. Ukuran Adapter: {adapter_size:.2f} MB")

In [34]:
# ============================================================
# CELL 8 — CONDITIONAL TRAINING STAGE (STANDALONE RUN)
# ============================================================

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import os
import torch

# 1. INISIALISASI STRUKTUR OUTPUT ADAPTER
exp_output_dir = os.path.join(CONFIG["output_dir"], CONFIG["experiment_name"])
os.makedirs(exp_output_dir, exist_ok=True)

print(f"⏳ Menyiapkan Base Model dan Arsitektur Adapter untuk {CONFIG['experiment_name']}...")
model = prepare_model_for_kbit_training(model)
model.config.torch_dtype = torch.float16

peft_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    target_modules="all-linear",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)

# Pembersihan sisa parameter BFloat16 demi menjaga stabilitas komputasi GPU Tesla T4
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16: param.data = param.data.to(torch.float16)
for name, buffer in model.named_buffers():
    if buffer.dtype == torch.bfloat16: buffer.data = buffer.data.to(torch.float16)

# 2. PROSES ROUTING CALLBACK SECARA DINAMIS
active_callbacks = []
if CONFIG["vocab_adaptation"] and (selective_callback is not None):
    active_callbacks.append(selective_callback)
    print("   -> 🧬 SelectiveEmbeddingCallback berhasil dipasang ke dalam Trainer.")

# 3. SETTING PARAMETER TRAINING ARGUMENTS
training_args = SFTConfig(
    output_dir=exp_output_dir,
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_acc_steps"],
    learning_rate=CONFIG["lr"],
    logging_steps=20,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    max_length=CONFIG["max_seq_length"],
    eval_strategy="steps",
    save_total_limit=2, 
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    report_to="none"
)

tokenizer.padding_side = "right"
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    processing_class=tokenizer,
    callbacks=active_callbacks  # Berisi list callback dinamis hasil router di atas
)

model.get_input_embeddings().weight.requires_grad = True

print(f"\n🚀 Memulai Proses Pelatihan QLoRA untuk Eksperimen: {CONFIG['experiment_name']}...")
trainer.train()

# Simpan model adapter hasil latihan
trainer.save_model(exp_output_dir)
print(f"✅ Adapter sukses disimpan di: {exp_output_dir}")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


⏳ Menyiapkan Base Model dan Arsitektur Adapter untuk C3_Norm_Morph...


Adding EOS to train dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

...


🚀 Memulai Proses Pelatihan QLoRA untuk Eksperimen: C3_Norm_Morph...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,1.225841,1.249172
200,1.233669,1.204870
300,1.222849,1.184145
400,1.165064,1.167753
500,1.073989,1.156280
600,1.129056,1.146489
700,1.126361,1.138251
800,1.085008,1.132444
900,1.071589,1.126184
1000,1.120506,1.120403


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

✅ Adapter sukses disimpan di: /kaggle/working/Riset_QLoRA/qwen-lora-adapter/C3_Norm_Morph


In [35]:
adapter_size = sum(os.path.getsize(os.path.join(CONFIG["output_dir"], f)) for f in os.listdir(CONFIG["output_dir"]) if os.path.isfile(os.path.join(CONFIG["output_dir"], f))) / (1024*1024)
print(f"✅ Training Selesai. Ukuran Adapter: {adapter_size:.2f} MB")

✅ Training Selesai. Ukuran Adapter: 0.00 MB


## Cell 9 — Eksperimen 1: Post-Training Evaluation & Analysis (old)

In [36]:
# print("""
# # ============================================================
# # CELL 9 — Eksperimen 1: Post-Training Evaluation & Analysis
# # ============================================================
# """)

In [37]:
# tokenizer.padding_side = "left"

In [38]:
# from peft import PeftModel

In [39]:
# comment dari awal

# from peft import PeftModel

# # Tentukan path tempat kamu menyimpan adapter kemarin
# # adapter_path = "/content/drive/MyDrive/Riset_QLoRA/qwen-lora-adapter"
# adapter_path = CONFIG['output_dir']

# print(f"🔄 Memuat Adapter LoRA dari: {adapter_path}...")

# # model di sini adalah base_model yang sudah kamu load di Cell 4
# model = PeftModel.from_pretrained(model, adapter_path)

# # Gabungkan secara permanen untuk mempercepat inferensi (Opsional tapi disarankan)
# model = model.merge_and_unload()

# model.eval()
# print("✅ Adapter Berhasil Terpasang! Model siap untuk Evaluasi Post-Training.")

In [40]:
# # ============================================================
# # 🛠️ PERBAIKAN LOAD MODEL UNTUK EVALUASI C4
# # ============================================================

# # 1. Muat tokenizer BARU dari folder hasil training C4 (buku kamus yang sudah diperluas)
# tokenizer = AutoTokenizer.from_pretrained(CONFIG['output_dir'])

# # 2. Muat Base Model Qwen asli dalam kondisi bersih (4-bit)
# base_model = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen2-1.5B",
#     quantization_config=bnb_config, # Mengganti load_in_4bit dengan quantization_config
#     device_map="auto",
#     trust_remote_code=True,
# )

# # 3. 🚨 LANGKAH KRUSIAL: Lebarkan ukuran embedding Base Model agar pas dengan tokenizer baru
# base_model.resize_token_embeddings(len(tokenizer))

# # 4. Sekarang tempelkan LoRA Adapter C4 ke Base Model yang sudah dilebarkan
# model = PeftModel.from_pretrained(base_model, CONFIG['output_dir']h)

# print("✅ Tokenizer baru dan Adapter C4 berhasil dimuat tanpa size mismatch!")

In [41]:
# # ============================================================
# # CELL 9 — Eksperimen 1: Post-Training Evaluation & Analysis
# # ============================================================
# import json
# import numpy as np
# import pandas as pd
# from sklearn.metrics import accuracy_score, f1_score

# print("=" * 60)
# print("EVALUASI POST-TRAINING (SESUDAH QLORA)")
# print("=" * 60)

# # 1. WAJIB: Balikkan padding ke LEFT untuk proses inferensi/generasi
# tokenizer.padding_side = "left"
# model.eval()

# post_results = {}
# # Tracker khusus metadata COPAL-ID
# copal_metadata_stats = {
#     "culture": {"1": {"correct": 0, "total": 0}, "0": {"correct": 0, "total": 0}},
#     "terminology": {"1": {"correct": 0, "total": 0}, "0": {"correct": 0, "total": 0}}
# }

# for task_name, dataset in eval_tasks.items():
#     if dataset is None: continue

#     print(f"\n🚀 Mengevaluasi {task_name.upper()}...")
#     eval_subset = dataset.select(range(min(CONFIG["eval_limit"], len(dataset))))

#     # Evaluasi Batch
#     raw_results = evaluate_batched_c3(model, tokenizer, eval_subset, task_name=task_name, batch_size=CONFIG["batch_size"])

#     preds, labels, latencies = [], [], []
#     unknown_count = 0

#     for i, res in enumerate(raw_results):
#         # 2. Gunakan Ekstraksi Hybrid (Pemaaf)
#         pred_final = extract_hybrid_prediction(task_name, res["pred"])

#         # 3. Gunakan Mapping Jujur
#         true_label = map_hybrid_label(task_name, eval_subset[i])

#         preds.append(pred_final)
#         labels.append(true_label)
#         latencies.append(res["latency"])

#         if pred_final == "UNKNOWN":
#             unknown_count += 1

#         # Tracker Metadata COPAL-ID
#         if task_name == "copal":
#             is_correct = (pred_final == true_label)
#             # Ambil dari kolom dengan huruf kapital
#             cult_val = str(eval_subset[i].get("Culture", "0"))
#             term_val = str(eval_subset[i].get("Terminology", "0"))

#             if cult_val in copal_metadata_stats["culture"]:
#                 copal_metadata_stats["culture"][cult_val]["total"] += 1
#                 if is_correct: copal_metadata_stats["culture"][cult_val]["correct"] += 1
#             if term_val in copal_metadata_stats["terminology"]:
#                 copal_metadata_stats["terminology"][term_val]["total"] += 1
#                 if is_correct: copal_metadata_stats["terminology"][term_val]["correct"] += 1

#     # 4. Kalkulasi Metrik (HONEST EVALUATION - Dihitung dari Total Keseluruhan)
#     acc = accuracy_score(labels, preds) * 100

#     # F1 Score dilewati untuk QA
#     if task_name == "qa":
#           # Gunakan metrik khusus QA (EM dan Token-F1)
#           em_scores = [qa_exact_match(p, l) for p, l in zip(preds, labels)]
#           f1_scores = [qa_f1_score(p, l) for p, l in zip(preds, labels)]

#           acc = np.mean(em_scores) * 100      # Exact Match diwakili sebagai Akurasi
#           mac_f1 = np.mean(f1_scores) * 100   # Token-F1 diwakili sebagai Macro F1
#     else:
#           # Gunakan metrik klasifikasi standar untuk COPAL, SmSA, IndoNLI, dll
#           acc = accuracy_score(labels, preds) * 100
#           mac_f1 = f1_score(labels, preds, average="macro", zero_division=0) * 100
#     avg_latency = np.mean(latencies)
#     print(f"    📊 Accuracy (Total) : {acc:.2f}%")
#     if task_name != "qa": print(f"    📊 Macro F1         : {mac_f1:.2f}%")
#     print(f"    ⚠️ Format Gagal     : {unknown_count} dari {len(preds)} sampel")

#     post_results[task_name] = {"accuracy": acc, "macro_f1": mac_f1, "latency": avg_latency, "fail": unknown_count}

# # Simpan hasil akhir
# with open("/kaggle/working/post_results.json", "w") as f:
#     json.dump(post_results, f, indent=4)

# # Simpan hasil akhir ke Google Drive
# # with open("/content/drive/MyDrive/Riset_QLoRA/post_results.json", "w") as f:
# #     json.dump(post_results, f, indent=4)

# # ============================================================
# # RINGKASAN PERBANDINGAN & ANALISIS
# # ============================================================
# print("\n" + "=" * 60)
# print("RINGKASAN PERBANDINGAN: BASELINE VS QLORA")
# print("=" * 60)

# try:
#     with open("/kaggle/working/baseline_results.json", "r") as f:
#         baseline_results = json.load(f)

#     # with open("/content/drive/MyDrive/Riset_QLoRA/baseline_results.json", "r") as f:
#     #     baseline_results = json.load(f)


#     for task in post_results:
#         b_acc = baseline_results.get(task, {}).get("accuracy", 0)
#         p_acc = post_results[task]["accuracy"]
#         delta = p_acc - b_acc
#         b_fail = baseline_results.get(task, {}).get("fail", 0)
#         p_fail = post_results[task]["fail"]

#         print(f"📌 {task.upper()}:")
#         print(f"   Akurasi      : {b_acc:.2f}% -> {p_acc:.2f}% ({delta:+.2f}%)")
#         print(f"   Gagal Format : {b_fail} -> {p_fail} sampel")
# except FileNotFoundError:
#     print("⚠️ File baseline_results.json tidak ditemukan.")

# # Cetak Analisis Metadata COPAL
# if "copal" in eval_tasks:
#     print("\n" + "=" * 60)
#     print("ANALISIS METADATA COPAL-ID")
#     print("=" * 60)
#     for meta_type, stats in copal_metadata_stats.items():
#         print(f"Berdasarkan {meta_type.capitalize()}:")
#         for val, s in stats.items():
#             score = (s["correct"]/s["total"]*100) if s["total"] > 0 else 0
#             label = "Ya (1)" if val == "1" else "Tidak (0)"
#             print(f" - {label}: {score:.2f}% ({s['correct']}/{s['total']})")

In [42]:
# # Simpan hasil POST-TRAINING
# with open("/kaggle/working/Riset_QLoRA/post_results.json", "w") as f:
#     json.dump(baseline_results, f, indent=4) # Pastikan nama variabel dict-nya sesuai
# print("\n✅ Evaluasi Post-QLoRA Selesai!")

## Cell 9 — Eksperimen 1: Post-Training Evaluation & Analysis (old)

In [43]:
# ============================================================
# CELL 9 — POST-TRAINING EVALUATION & ANALYSIS (PRESERVED)
# ============================================================
import json
import numpy as np
import pandas as pd
import os
from sklearn.metrics import accuracy_score, f1_score
from peft import PeftModel

print("=" * 60)
print(f"EVALUASI POST-TRAINING UNTUK: {CONFIG['experiment_name']}")
print("=" * 60)

# 1. TEMPAT PENYIMPANAN ADAPTER SESUAI SAKLAR AKTIF
exp_output_dir = os.path.join(CONFIG["output_dir"], CONFIG["experiment_name"])



EVALUASI POST-TRAINING UNTUK: C3_Norm_Morph


In [44]:
# 2. SEAMLESS MODEL RELOADING (Memastikan State Tokenizer & Embedding Sinkron)
print(f"⏳ Memuat ulang Tokenizer dan Adapter dari: {exp_output_dir}...")
tokenizer = AutoTokenizer.from_pretrained(exp_output_dir)
tokenizer.padding_side = "left" # Wajib LEFT padding untuk proses inferensi/generasi

base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Lebarkan ukuran embedding Base Model agar pas dengan kamus baru hasil training
base_model.resize_token_embeddings(len(tokenizer))

# Tempelkan LoRA Adapter khusus milik eksperimen aktif saat ini
model = PeftModel.from_pretrained(base_model, exp_output_dir)
model.eval()
print("✅ Tokenizer baru dan Adapter berhasil dimuat ke memori tanpa size mismatch!")

⏳ Memuat ulang Tokenizer dan Adapter dari: /kaggle/working/Riset_QLoRA/qwen-lora-adapter/C3_Norm_Morph...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Tokenizer baru dan Adapter berhasil dimuat ke memori tanpa size mismatch!


In [45]:
# 3. INISIALISASI TRACKER METADATA ASLI ANDA
post_results = {}
copal_metadata_stats = {
    "culture": {"1": {"correct": 0, "total": 0}, "0": {"correct": 0, "total": 0}},
    "terminology": {"1": {"correct": 0, "total": 0}, "0": {"correct": 0, "total": 0}}
}



In [46]:
# 4. LOOP INFERENSI DOWNSTREAM TASKS
for task_name, dataset in eval_tasks.items():
    if dataset is None: continue

    print(f"\n🚀 Mengevaluasi {task_name.upper()}...")
    eval_subset = dataset.select(range(min(CONFIG["eval_limit"], len(dataset))))

    # Memanggil mesin inferensi batch tunggal dari Cell 6 yang baru (Otomatis membaca saklar tag)
    raw_results = evaluate_batched_final(
        model=model, 
        tokenizer=tokenizer, 
        dataset=eval_subset, 
        task_name=task_name, 
        linguistic_mode=CONFIG["linguistic_mode"], 
        batch_size=CONFIG["batch_size"]
    )

    preds, labels, latencies = [], [], []
    unknown_count = 0

    for i, res in enumerate(raw_results):
        pred_final = extract_hybrid_prediction(task_name, res["pred"])
        true_label = map_hybrid_label(task_name, eval_subset[i])

        preds.append(pred_final)
        labels.append(true_label)
        latencies.append(res["latency"])

        if pred_final == "UNKNOWN":
            unknown_count += 1

        # TRACKER METADATA COPAL-ID ASLI ANDA (DIPERTAHANKAN 100%)
        if task_name == "copal":
            is_correct = (pred_final == true_label)
            cult_val = str(eval_subset[i].get("Culture", "0"))
            term_val = str(eval_subset[i].get("Terminology", "0"))

            if cult_val in copal_metadata_stats["culture"]:
                copal_metadata_stats["culture"][cult_val]["total"] += 1
                if is_correct: copal_metadata_stats["culture"][cult_val]["correct"] += 1
            if term_val in copal_metadata_stats["terminology"]:
                copal_metadata_stats["terminology"][term_val]["total"] += 1
                if is_correct: copal_metadata_stats["terminology"][term_val]["correct"] += 1

    # 5. KALKULASI METRIK EVALUASI
    if task_name == "qa":
        em_scores = [qa_exact_match(p, l) for p, l in zip(preds, labels)]
        f1_scores = [qa_f1_score(p, l) for p, l in zip(preds, labels)]
        acc = np.mean(em_scores) * 100      
        mac_f1 = np.mean(f1_scores) * 100   
    else:
        acc = accuracy_score(labels, preds) * 100
        unique_labels = list(set(labels))
        mac_f1 = f1_score(labels, preds, average="macro", zero_division=0) * 100
        
    avg_latency = np.mean(latencies)
    print(f"    📊 Accuracy (Total) : {acc:.2f}%")
    if task_name != "qa": print(f"    📊 Macro F1         : {mac_f1:.2f}%")
    print(f"    ⚠️ Format Gagal     : {unknown_count} dari {len(preds)} sampel")

    post_results[task_name] = {"accuracy": acc, "macro_f1": mac_f1, "latency": avg_latency, "fail": unknown_count}




🚀 Mengevaluasi COPAL...


Eval copal: 100%|██████████| 75/75 [01:51<00:00,  1.48s/it]


    📊 Accuracy (Total) : 58.67%
    📊 Macro F1         : 40.19%
    ⚠️ Format Gagal     : 8 dari 150 sampel

🚀 Mengevaluasi SMSA...


Eval smsa: 100%|██████████| 75/75 [01:07<00:00,  1.12it/s]


    📊 Accuracy (Total) : 94.67%
    📊 Macro F1         : 88.93%
    ⚠️ Format Gagal     : 0 dari 150 sampel

🚀 Mengevaluasi INDONLI...


Eval indonli: 100%|██████████| 75/75 [01:12<00:00,  1.03it/s]


    📊 Accuracy (Total) : 58.00%
    📊 Macro F1         : 55.20%
    ⚠️ Format Gagal     : 0 dari 150 sampel

🚀 Mengevaluasi INDOCULTURE...


Eval indoculture: 100%|██████████| 75/75 [01:52<00:00,  1.50s/it]

    📊 Accuracy (Total) : 44.67%
    📊 Macro F1         : 40.36%
    ⚠️ Format Gagal     : 0 dari 150 sampel


In [47]:
# 6. PENYIMPANAN LOG JSON (Dibuat unik per eksperimen agar tidak saling menimpa)
json_out_path = f"/kaggle/working/post_results_{CONFIG['experiment_name']}.json"
with open(json_out_path, "w") as f:
    json.dump(post_results, f, indent=4)
print(f"\n📝 File log hasil berhasil disimpan di: {json_out_path}")


# ============================================================
# 7. RINGKASAN PERBANDINGAN & ANALISIS DELTA (PRESERVED)
# ============================================================
print("\n" + "=" * 60)
print(f"RINGKASAN PERBANDINGAN: BASELINE VS {CONFIG['experiment_name']}")
print("=" * 60)

try:
    with open("/kaggle/working/baseline_results.json", "r") as f:
        baseline_results = json.load(f)

    for task in post_results:
        b_acc = baseline_results.get(task, {}).get("accuracy", 0)
        p_acc = post_results[task]["accuracy"]
        delta = p_acc - b_acc
        b_fail = baseline_results.get(task, {}).get("fail", 0)
        p_fail = post_results[task]["fail"]

        print(f"📌 {task.upper()}:")
        print(f"   Akurasi      : {b_acc:.2f}% -> {p_acc:.2f}% ({delta:+.2f}%)")
        print(f"   Gagal Format : {b_fail} -> {p_fail} sampel")
except FileNotFoundError:
    print("⚠️ File baseline_results.json tidak ditemukan untuk analisis delta.")





📝 File log hasil berhasil disimpan di: /kaggle/working/post_results_C3_Norm_Morph.json

RINGKASAN PERBANDINGAN: BASELINE VS C3_Norm_Morph
⚠️ File baseline_results.json tidak ditemukan untuk analisis delta.


In [48]:
# ============================================================
# 8. OUTPUT CETAK METADATA COPAL-ID (PRESERVED)
# ============================================================
if "copal" in eval_tasks:
    print("\n" + "=" * 60)
    print("ANALISIS METADATA COPAL-ID")
    print("=" * 60)
    for meta_type, stats in copal_metadata_stats.items():
        print(f"Berdasarkan {meta_type.capitalize()}:")
        for val, s in stats.items():
            score = (s["correct"]/s["total"]*100) if s["total"] > 0 else 0
            label = "Ya (1)" if val == "1" else "Tidak (0)"
            print(f" - {label}: {score:.2f}% ({s['correct']}/{s['total']})")

# 9. PERBAIKAN BUG SAVING DI BARIS AKHIR (Menyimpan post_results ke workspace)
final_backup_path = f"/kaggle/working/post_results_{CONFIG['experiment_name']}_backup.json"
with open(final_backup_path, "w") as f:
    json.dump(post_results, f, indent=4) # FIX: Mengubah dari baseline_results ke post_results
    
print(f"\n✅ Selesai! Seluruh Rangkaian Evaluasi {CONFIG['experiment_name']} Telah Berhasil.")


ANALISIS METADATA COPAL-ID
Berdasarkan Culture:
 - Ya (1): 60.61% (40/66)
 - Tidak (0): 57.14% (48/84)
Berdasarkan Terminology:
 - Ya (1): 57.61% (53/92)
 - Tidak (0): 60.34% (35/58)

✅ Selesai! Seluruh Rangkaian Evaluasi C3_Norm_Morph Telah Berhasil.


In [49]:
# ====================================================================
# 📊 CELL DIAGNOSTIK 1: ANALISIS FRAGMENTASI TOKEN (FERTILITY) PER BENCHMARK
# ====================================================================
import re
import pandas as pd
import numpy as np

print("⏳ Menghitung karakteristik tokenisasi pada setiap benchmark dataset...")
fertility_report = []

for task_name, dataset in eval_tasks.items():
    if dataset is None: continue
        
    all_word_fertilities = []
    high_fragmentation_count = 0
    total_words_scanned = 0
    
    for sample in dataset:
        if task_name == "copal":
            text = f"{sample['premise']} {sample['choice1']} {sample['choice2']}"
        elif task_name == "qa":
            text = " ".join(sample.get("passage", []))
        elif task_name == "indoculture":
            text = f"{sample['context']} " + " ".join(sample.get("options", []))
        elif task_name == "indonli":
            text = f"{sample['premise']} {sample['hypothesis']}"
        else: # smsa
            text = sample.get("text", sample.get("sentence", ""))
            
        words = re.findall(r'\b[a-zA-Z]{4,}\b', str(text).lower())
        for word in words:
            total_words_scanned += 1
            n_subwords = len(tokenizer.encode(word, add_special_tokens=False))
            all_word_fertilities.append(n_subwords)
            if n_subwords >= 3:
                high_fragmentation_count += 1
                
    avg_fertility = np.mean(all_word_fertilities) if all_word_fertilities else 0
    pct_fragmented = (high_fragmentation_count / total_words_scanned * 100) if total_words_scanned > 0 else 0
    
    fertility_report.append({
        "Dataset Benchmark": task_name.upper(),
        "Rata-rata Subword/Kata (Fertility)": round(avg_fertility, 2),
        "Persentase Kata Hancur (Fertility >= 3) (%)": round(pct_fragmented, 2),
        "Total Kata yang Diperiksa": total_words_scanned
    })

df_fertility = pd.DataFrame(fertility_report)
print("\n" + "="*70)
print("📊 TABEL KORELASI: KARAKTERISTIK TOKENISASI PER BENCHMARK DATASET")
print("="*70)
print(df_fertility.to_string(index=False))

⏳ Menghitung karakteristik tokenisasi pada setiap benchmark dataset...

📊 TABEL KORELASI: KARAKTERISTIK TOKENISASI PER BENCHMARK DATASET
Dataset Benchmark  Rata-rata Subword/Kata (Fertility)  Persentase Kata Hancur (Fertility >= 3) (%)  Total Kata yang Diperiksa
            COPAL                                2.33                                        35.10                       7399
             SMSA                                2.20                                        26.66                       8786
          INDONLI                                2.37                                        35.94                     104707
      INDOCULTURE                                2.32                                        32.58                      19216


In [50]:
# ====================================================================
# 📝 CELL DIAGNOSTIK 2: INSPEKSI KUALITATIF TOP KANDIDAT TOKEN BARU
# ====================================================================

if CONFIG["vocab_adaptation"] and 'fertility_stats' in locals():
    print("\n" + "="*70)
    print("📝 ANALISIS KUALITATIF: 30 TOKEN BARU TERATAS YANG BERHASIL DIADAPTASI")
    print("="*70)
    
    # Mengambil 30 kata teratas yang lolos fertility threshold dari korpus Anda
    top_added_tokens = fertility_stats[:30]
    
    formatted_analysis = []
    for idx, stat in enumerate(top_added_tokens, 1):
        formatted_analysis.append({
            "No": idx,
            "Token Baru": stat["word"],
            "Frekuensi Korpus": stat["freq"],
            "Jumlah Subword Asli": stat["fertility"]
        })
        
    df_top_tokens = pd.DataFrame(formatted_analysis)
    print(df_top_tokens.to_string(index=False))
else:
    print("\n⏩ Lewati diagnostik 2: Aktifkan 'vocab_adaptation': True terlebih dahulu pada CONFIG untuk melihat hasil perluasan token.")


⏩ Lewati diagnostik 2: Aktifkan 'vocab_adaptation': True terlebih dahulu pada CONFIG untuk melihat hasil perluasan token.
